# Chunking de manifestações longas
Comparação de duas configurações com `RecursiveCharacterTextSplitter`.

In [1]:
import pandas as pd, plotly.express as px
from ouvidoria_core import *
base=load_manifestacoes(); longas=sorted(base,key=lambda x:len(x['texto']),reverse=True)[:5]
[(x['id'],len(x['texto'])) for x in longas]

[('M033', 675), ('M034', 623), ('M035', 617), ('M036', 594), ('M037', 593)]

In [2]:
configs=[(300,50),(450,80)]
for size,overlap in configs:
    print(f'Configuração {size}/{overlap}')
    for item in longas:
        print(item['id'], split_text(item['texto'], size, overlap))

Configuração 300/50


M033 ['A Avenida Brasil sofre com crateras há meses. Em frente ao terminal, uma abertura enorme já danificou carros e motos. À noite ela não é visível porque a sinalização sumiu. Pedimos que a equipe faça o tapa-buraco e revise toda a pista, pois ônibus e ambulâncias passam por ali diariamente', '. Nos dias de chuva a água enche a cratera e o desvio força veículos a entrar na faixa contrária, aumentando muito o risco de acidente. Comerciantes do entorno relatam que o problema piorou após a última chuva e já protocolaram pedidos sem resposta', '. Também é necessário instalar sinalização provisória até o reparo definitivo, porque crianças atravessam a avenida para chegar ao terminal.']
M034 ['Na UBS do Jardim não há clínico geral em vários dias da semana. Minha mãe, idosa e hipertensa, chegou às seis horas e recebeu a informação de que não haveria consulta. Outros pacientes disseram que isso acontece repetidamente. Quando há médico, as vagas acabam antes das sete', '. Quando há médico, a

In [3]:
linhas=[]
for item in longas:
    for n,chunk in enumerate(split_text(item['texto'],450,80),1): linhas.append({'manifestacao':item['id'],'chunk':n,'texto':chunk})
df=pd.DataFrame(linhas); xy=reduce_embeddings(embed_texts(df.texto), 'PCA'); df[['x','y']]=xy
px.scatter(df,x='x',y='y',color='manifestacao',hover_data=['chunk','texto'],title='Chunks no espaço PCA')

**Discussão.** O overlap repete a fronteira e mantém referentes, locais e consequências entre chunks consecutivos. Em geral 450/80 preserva melhor a narrativa; 300/50 é mais granular e pode separar uma solicitação de sua justificativa. A proximidade por documento no gráfico sugere coesão, mas deve ser verificada lendo os trechos.